# Six-frequency FBTDCA training

Train and validate a participant-specific 8/9/10/13/14/15 Hz model from at least two complete collection sessions. Epochs span 0.25–2.25 seconds after confirmed stimulus onset.

In [ ]:
from pathlib import Path
import os
import matplotlib.pyplot as plt
import numpy as np
from ssvep_bci.training import (
    discover_collection_sessions, load_fbtdca_dataset,
    save_fbtdca_model, train_fbtdca,
)

def find_repo_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "ssvep-bci" / "pyproject.toml").is_file():
            return candidate
        if (candidate / "src" / "ssvep_bci").is_dir():
            return candidate.parent
    raise FileNotFoundError("Could not locate the BCI repository root")

REPO_ROOT = find_repo_root()
EXPERIMENT_ID = "six-frequency-fbtdca-collection"
PARTICIPANT_ID = os.getenv("SSVEP_PARTICIPANT_ID", "replace_me")
SESSION_ROOT = Path(os.getenv("SSVEP_SESSION_ROOT", str(REPO_ROOT / "ssvep-bci/sessions")))
MODEL_PATH = Path(os.getenv("SSVEP_MODEL_PATH", str(REPO_ROOT / "ssvep-bci/models" / PARTICIPANT_ID / "six_frequency_fbtdca.joblib")))

## Discover, validate, and extract timestamp-aligned epochs

In [ ]:
session_paths = discover_collection_sessions(
    SESSION_ROOT, PARTICIPANT_ID, experiment_id=EXPERIMENT_ID
)
dataset = load_fbtdca_dataset(
    session_paths, PARTICIPANT_ID, experiment_id=EXPERIMENT_ID
)
class_count = len(dataset.candidate_frequencies_hz)
print("Sessions:", len(dataset.session_paths))
print("EEG shape (trials, channels, samples):", dataset.eeg.shape)
print("Window:", dataset.window_onset_offset_seconds, "to",
      dataset.window_onset_offset_seconds + dataset.window_length_seconds, "seconds")
print("Frequencies:", dataset.candidate_frequencies_hz)
print("Excluded incomplete trials:", len(dataset.excluded_trials))
print("Trials per class:", np.bincount(dataset.labels, minlength=class_count))

## Leave-one-session-out validation and final fit

In [ ]:
estimator, validation = train_fbtdca(dataset)
for fold in validation["folds"]:
    print(f"{fold['held_out_session_id']}: balanced accuracy {fold['balanced_accuracy']:.3f}")
print(f"Overall accuracy: {validation['accuracy']:.3f}")
print(f"Overall balanced accuracy: {validation['balanced_accuracy']:.3f}")

In [ ]:
matrix = np.asarray(validation["normalized_confusion_matrix"])
fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(matrix, cmap="Blues", vmin=0, vmax=1)
frequency_labels = [f"{value:g} Hz" for value in dataset.candidate_frequencies_hz]
indexes = range(class_count)
ax.set(xticks=indexes, yticks=indexes, xticklabels=frequency_labels,
       yticklabels=frequency_labels, xlabel="Predicted",
       ylabel="Attended target",
       title="Six-frequency FBTDCA leave-one-session-out confusion matrix")
for row in indexes:
    for column in indexes:
        ax.text(column, row, f"{matrix[row, column]:.2f}", ha="center", va="center")
fig.colorbar(image, ax=ax)
fig.tight_layout()
plt.show()

## Save the final estimator and reproducibility metadata

In [ ]:
saved_model, saved_summary = save_fbtdca_model(
    dataset, estimator, validation, MODEL_PATH
)
print("Saved model:", saved_model)
print("Saved summary:", saved_summary)